In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("asset_id", "")
dbutils.widgets.text("assignment_id", "")
dbutils.widgets.text("rule_code", "")
dbutils.widgets.text("version_number", "")
dbutils.widgets.text("target_column_id", "")
dbutils.widgets.text("asset_execution_id", "")
dbutils.widgets.text("parameter_values", "")
dbutils.widgets.text("sql_expression", "")
dbutils.widgets.text("threshold_value", "")
dbutils.widgets.text("evaluation_operation", "")


asset_id = dbutils.widgets.get("asset_id").strip()
assignment_id = dbutils.widgets.get("assignment_id").strip()
rule_code = dbutils.widgets.get("rule_code").strip()
version_number = dbutils.widgets.get("version_number").strip()
target_column_id = dbutils.widgets.get("target_column_id").strip()
asset_execution_id = dbutils.widgets.get(
    "asset_execution_id"
).strip()
parameter_values = dbutils.widgets.get("parameter_values").strip()
sql_expression = dbutils.widgets.get("sql_expression").strip()
threshold_value = dbutils.widgets.get("threshold_value").strip()
evaluation_operation = dbutils.widgets.get("evaluation_operation").strip()


print("====================================")
print("RULE EXECUTION INPUT")
print("====================================")

print(f"asset_id             : {asset_id}")
print(f"assignment_id        : {assignment_id}")
print(f"rule_code            : {rule_code}")
print(f"version_number       : {version_number}")
print(f"target_column_id     : {target_column_id}")
print(f"parameter_values     : {parameter_values}")
print(f"sql_expression       : {sql_expression}")
print(f"threshold_value      : {threshold_value}")
print(f"evaluation_operation : {evaluation_operation}")

RULE EXECUTION INPUT
asset_id             : 28
assignment_id        : 35
rule_code            : T03
version_number       : 1
target_column_id     : 
parameter_values     : {"source_timestamp_column":"INITIATED_AT","ingestion_timestamp_column":"SETTLED_AT","time_unit":"MINUTE"}
sql_expression       : 
threshold_value      : 60
evaluation_operation : <=


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import json
from decimal import Decimal
import pandas as pd
import re


CATALOG = "mastercard_card_241614"
SCHEMA = "default"

RULES_TABLE = f"{CATALOG}.{SCHEMA}.rules_m"
ASSIGNMENTS_TABLE = f"{CATALOG}.{SCHEMA}.rules_assignments_m"
ASSETS_TABLE = f"{CATALOG}.{SCHEMA}.assets_m"
ASSET_COLUMNS_TABLE = f"{CATALOG}.{SCHEMA}.asset_columns_m"

SOURCE_EXCEL = "/Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx"

SOURCE_VIEW = "dq_source_data"


print("====================================")
print("RULE EXECUTION CONFIGURATION")
print("====================================")

print(f"Rules table        : {RULES_TABLE}")
print(f"Assignments table  : {ASSIGNMENTS_TABLE}")
print(f"Assets table       : {ASSETS_TABLE}")
print(f"Columns table      : {ASSET_COLUMNS_TABLE}")
print(f"Source Excel       : {SOURCE_EXCEL}")
print(f"Source temp view   : {SOURCE_VIEW}")

RULE EXECUTION CONFIGURATION
Rules table        : mastercard_card_241614.default.rules_m
Assignments table  : mastercard_card_241614.default.rules_assignments_m
Assets table       : mastercard_card_241614.default.assets_m
Columns table      : mastercard_card_241614.default.asset_columns_m
Source Excel       : /Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx
Source temp view   : dq_source_data


In [0]:
print("====================================")
print("VALIDATING RULE EXECUTION INPUT")
print("====================================")

required_inputs = {
    "asset_id": asset_id,
    "assignment_id": assignment_id,
    "rule_code": rule_code,
    "version_number": version_number
}

missing_inputs = [
    name
    for name, value in required_inputs.items()
    if value == ""
]

if missing_inputs:
    raise ValueError(
        f"Missing required rule execution inputs: {missing_inputs}"
    )


try:
    asset_id_int = int(asset_id)
    assignment_id_int = int(assignment_id)
    version_number_int = int(version_number)
except ValueError:
    raise ValueError(
        "asset_id, assignment_id and version_number must be valid integers."
    )


if target_column_id != "":
    try:
        target_column_id_int = int(target_column_id)
    except ValueError:
        raise ValueError(
            "target_column_id must be a valid integer when provided."
        )
else:
    target_column_id_int = None


print("Input validation PASSED")
print(f"asset_id         : {asset_id_int}")
print(f"assignment_id    : {assignment_id_int}")
print(f"rule_code        : {rule_code}")
print(f"version_number   : {version_number_int}")
print(f"target_column_id : {target_column_id_int}")

VALIDATING RULE EXECUTION INPUT
Input validation PASSED
asset_id         : 28
assignment_id    : 35
rule_code        : T03
version_number   : 1
target_column_id : None


In [0]:
print("====================================")
print("RESOLVING RULE ASSIGNMENT")
print("====================================")


rules_assignments_df = spark.table(ASSIGNMENTS_TABLE)


assignment_df = (
    rules_assignments_df
    .filter(F.col("assignment_id") == assignment_id_int)
    .filter(F.col("asset_id") == asset_id_int)
    .filter(F.col("rule_code") == rule_code)
    .filter(F.col("version_number") == version_number_int)
)


assignment_count = assignment_df.count()


if assignment_count == 0:
    raise ValueError(
        "No matching rule assignment found.\n"
        f"asset_id={asset_id_int}, "
        f"assignment_id={assignment_id_int}, "
        f"rule_code={rule_code}, "
        f"version_number={version_number_int}"
    )


if assignment_count > 1:
    raise ValueError(
        "Multiple matching rule assignments found. "
        "assignment_id should uniquely identify an assignment."
    )


assignment = assignment_df.first()


print("Rule assignment resolved successfully")
print("------------------------------------")

print(f"assignment_id        : {assignment['assignment_id']}")
print(f"asset_id             : {assignment['asset_id']}")
print(f"rule_code            : {assignment['rule_code']}")
print(f"version_number       : {assignment['version_number']}")
print(f"target_column_id     : {assignment['target_column_id']}")
print(f"target_type          : {assignment['target_type']}")
print(f"parameter_values     : {assignment['parameter_values']}")
print(f"sql_expression       : {assignment['sql_expression']}")
print(f"threshold_value      : {assignment['threshold_value']}")
print(f"evaluation_operation : {assignment['evaluation_operation']}")
print(f"is_enabled           : {assignment['is_enabled']}")
print(f"is_current           : {assignment['is_current']}")
# print("====================================")
# print("CHECKING CURRENT RULE ASSIGNMENTS")
# print("====================================")

# rules_assignments_df = spark.table(ASSIGNMENTS_TABLE)

# display(
#     rules_assignments_df
#     .filter(F.col("asset_id") == 28)
#     .filter(F.col("rule_code") == "C01")
#     .select(
#         "assignment_id",
#         "asset_id",
#         "rule_code",
#         "version_number",
#         "target_column_id",
#         "target_type",
#         "parameter_values",
#         "sql_expression",
#         "threshold_value",
#         "evaluation_operation",
#         "is_enabled",
#         "is_current",
#         "effective_from",
#         "effective_to"
#     )
#     .orderBy("assignment_id")
# )

RESOLVING RULE ASSIGNMENT
Rule assignment resolved successfully
------------------------------------
assignment_id        : 35
asset_id             : 28
rule_code            : T03
version_number       : 1
target_column_id     : None
target_type          : TABLE / COLUMN
parameter_values     : None
sql_expression       : None
threshold_value      : None
evaluation_operation : <=
is_enabled           : True
is_current           : True


In [0]:
print("====================================")
print("RESOLVING RULE METADATA")
print("====================================")


rules_df = spark.table(RULES_TABLE)


rule_metadata_df = (
    rules_df
    .filter(F.col("rule_code") == rule_code)
    .filter(F.col("is_active") == True)
)


rule_metadata_count = rule_metadata_df.count()


if rule_metadata_count == 0:
    raise ValueError(
        f"No active rule metadata found for rule_code='{rule_code}'."
    )


if rule_metadata_count > 1:
    raise ValueError(
        f"Multiple active rule definitions found for "
        f"rule_code='{rule_code}'."
    )


rule_metadata = rule_metadata_df.first()


dimension = rule_metadata["dimension"]
rule_name = rule_metadata["rule_name"]
rule_type = rule_metadata["rule_type"]
description = rule_metadata["description"]
target_type = rule_metadata["target_type"]
parameter_definition = rule_metadata["parameter_definition"]
sql_query_template = rule_metadata["sql_query_template"]
default_threshold = rule_metadata["default_threshold"]
data_type = rule_metadata["data_type"]
semantic_type = rule_metadata["semantic_type"]


print("Rule metadata resolved successfully")
print("------------------------------------")

print(f"rule_code            : {rule_metadata['rule_code']}")
print(f"dimension            : {dimension}")
print(f"rule_name            : {rule_name}")
print(f"rule_type            : {rule_type}")
print(f"target_type          : {target_type}")
print(f"data_type            : {data_type}")
print(f"semantic_type        : {semantic_type}")
print(f"default_threshold    : {default_threshold}")
print(f"parameter_definition : {parameter_definition}")
print("------------------------------------")
print("SQL QUERY TEMPLATE")
print("------------------------------------")
print(sql_query_template)

RESOLVING RULE METADATA
Rule metadata resolved successfully
------------------------------------
rule_code            : T03
dimension            : TIMELINESS
rule_name            : Average latency
rule_type            : PARAMETERIZED
target_type          : TABLE / COLUMN
data_type            : DATE/TIMESTAMP
semantic_type        : DATE_TIME
default_threshold    : None
parameter_definition : {
          "parameters":[
            {
              "name":"source_timestamp_column",
              "label":"Source timestamp column",
              "control":"column_select",
              "required":true
            },
            {
              "name":"ingestion_timestamp_column",
              "label":"Ingestion timestamp column",
              "control":"column_select",
              "required":true
            },
            {
              "name":"time_unit",
              "label":"Time unit",
              "control":"select",
              "options":["SECOND","MINUTE","HOUR","DAY"],
    

In [0]:
print("====================================")
print("RESOLVING ASSET AND TARGET COLUMN")
print("====================================")


# --------------------------------------------------
# 1. Load asset metadata
# --------------------------------------------------

assets_df = spark.table(ASSETS_TABLE)

asset_df = (
    assets_df
    .filter(F.col("asset_id") == asset_id_int)
    .filter(F.col("is_active") == True)
)


asset_count = asset_df.count()


if asset_count == 0:
    raise ValueError(
        f"No active asset found for asset_id={asset_id_int}."
    )


if asset_count > 1:
    raise ValueError(
        f"Multiple active assets found for asset_id={asset_id_int}."
    )


asset = asset_df.first()

asset_name = asset["asset_name"]
asset_type = asset["asset_type"]
asset_s3_path = asset["s3_path"]
asset_domain_id = asset["domain_id"]
asset_sub_domain_id = asset["sub_domain_id"]


print("Asset resolved")
print("------------------------------------")
print(f"asset_id       : {asset_id_int}")
print(f"asset_name     : {asset_name}")
print(f"asset_type     : {asset_type}")
print(f"s3_path        : {asset_s3_path}")
print(f"domain_id      : {asset_domain_id}")
print(f"sub_domain_id  : {asset_sub_domain_id}")


# --------------------------------------------------
# 2. Resolve target column when provided
# --------------------------------------------------

target_column_name = None
target_column_data_type = None
target_column_nullable = None
target_column_metadata = None


if target_column_id_int is not None:

    asset_columns_df = spark.table(ASSET_COLUMNS_TABLE)

    target_column_df = (
        asset_columns_df
        .filter(F.col("column_id") == target_column_id_int)
        .filter(F.col("asset_id") == asset_id_int)
    )


    target_column_count = target_column_df.count()


    if target_column_count == 0:
        raise ValueError(
            "Target column could not be resolved.\n"
            f"column_id={target_column_id_int}, "
            f"asset_id={asset_id_int}"
        )


    if target_column_count > 1:
        raise ValueError(
            "Multiple target columns found for the same "
            "column_id and asset_id."
        )


    target_column = target_column_df.first()

    target_column_name = target_column["column_name"]
    target_column_data_type = target_column["data_type"]
    target_column_nullable = target_column["is_nullable"]
    target_column_metadata = target_column["metadata"]


    print("")
    print("Target column resolved")
    print("------------------------------------")
    print(f"column_id       : {target_column_id_int}")
    print(f"column_name     : {target_column_name}")
    print(f"data_type       : {target_column_data_type}")
    print(f"is_nullable     : {target_column_nullable}")
    print(f"metadata        : {target_column_metadata}")

else:

    print("")
    print("No target column specified.")
    print("Rule will execute at asset/table level.")

RESOLVING ASSET AND TARGET COLUMN
Asset resolved
------------------------------------
asset_id       : 28
asset_name     : 01_TXN_transaction_lifecycle
asset_type     : TABLE
s3_path        : source/TXN_PROCESSING/TRANSACTION_LIFECYCLE_STATUS/01_txn_transaction_lifecycle
domain_id      : 1
sub_domain_id  : 1

No target column specified.
Rule will execute at asset/table level.


In [0]:
# ============================================================
# VALIDATE TARGET COLUMN
# ============================================================

if target_column_id_int is not None:

    target_column_rows = (
        asset_columns_df
        .filter(
            (F.col("asset_id") == asset_id_int) &
            (F.col("column_id") == target_column_id_int)
        )
        .collect()
    )

    if not target_column_rows:

        raise ValueError(
            f"Target column ID {target_column_id_int} "
            f"does not exist for asset_id {asset_id_int}. "
            f"Assignment {assignment_id_int} "
            f"must be corrected in rules_assignments_m."
        )

    target_column_name = target_column_rows[0]["column_name"]

    print(
        f"Target column resolved : "
        f"{target_column_id_int} -> {target_column_name}"
    )

else:

    target_column_name = None

    print("Rule is table-level.")

Rule is table-level.


In [0]:
# ============================================================
# LOADING SOURCE DATA
# ============================================================

print("====================================")
print("LOADING SOURCE DATA")
print("====================================")


# --------------------------------------------------
# 1. Load the asset's Excel sheet
# --------------------------------------------------

print(f"Excel file : {SOURCE_EXCEL}")
print(f"Sheet name : {asset_name}")

pdf = pd.read_excel(
    SOURCE_EXCEL,
    sheet_name=asset_name
)

print(f"Rows loaded from Excel : {len(pdf)}")
print(f"Columns found           : {len(pdf.columns)}")


# --------------------------------------------------
# 2. Preserve raw source values
# --------------------------------------------------
#
# All source values are intentionally kept as strings.
#
# This is important because DQ rules must be able to
# distinguish:
#
#   - valid typed values
#   - malformed values
#   - NULL values
#
# For example:
#
#   "2023-03-09 07:40:53" -> valid timestamp
#   "TXN6022768040"        -> malformed timestamp
#   NULL                   -> missing value
#
# --------------------------------------------------

for col in pdf.columns:
    pdf[col] = pdf[col].apply(
        lambda x: str(x) if pd.notna(x) else None
    )


df = spark.createDataFrame(pdf)


# --------------------------------------------------
# 3. Basic source information
# --------------------------------------------------

print("")
print("Spark source DataFrame created")
print("------------------------------------")

print(f"Row count    : {df.count()}")
print(f"Column count : {len(df.columns)}")


# --------------------------------------------------
# 4. Register raw source view
# --------------------------------------------------

df.createOrReplaceTempView(SOURCE_VIEW)


print("")
print(f"Temporary view created : {SOURCE_VIEW}")


# --------------------------------------------------
# 5. Display source structure
# --------------------------------------------------

print("")
print("SOURCE SCHEMA")
print("------------------------------------")

df.printSchema()

LOADING SOURCE DATA
Excel file : /Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx
Sheet name : 01_TXN_transaction_lifecycle
Rows loaded from Excel : 100
Columns found           : 14

Spark source DataFrame created
------------------------------------
Row count    : 100
Column count : 14

Temporary view created : dq_source_data

SOURCE SCHEMA
------------------------------------
root
 |-- TRANSACTION_ID: string (nullable = true)
 |-- LIFECYCLE_STAGE: string (nullable = true)
 |-- TRANSACTION_TYPE: string (nullable = true)
 |-- AMOUNT: string (nullable = true)
 |-- CURRENCY_CODE: string (nullable = true)
 |-- INITIATED_AT: string (nullable = true)
 |-- AUTHORIZED_AT: string (nullable = true)
 |-- CLEARED_AT: string (nullable = true)
 |-- SETTLED_AT: string (nullable = true)
 |-- CHANNEL: string (nullable = true)
 |-- IS_CROSS_BORDER: string (nullable = true)
 |-- NETWORK: string (nullable = true)
 |-- PAN_LAST4: string (nullable = true)
 |-- DQ_FLAG: string (

In [0]:
print("====================================")
print("PARSING RULE PARAMETERS")
print("====================================")


# --------------------------------------------------
# 1. Initialize parameter values
# --------------------------------------------------

parsed_parameters = {}


if parameter_values in ("", "null", "NULL", "None"):
    parsed_parameters = {}

else:
    try:
        parsed_parameters = json.loads(parameter_values)

    except json.JSONDecodeError as e:
        raise ValueError(
            "parameter_values is not valid JSON.\n"
            f"Received: {parameter_values}\n"
            f"Error: {str(e)}"
        )


# --------------------------------------------------
# 2. Validate parameter structure
# --------------------------------------------------

if not isinstance(parsed_parameters, dict):
    raise ValueError(
        "parameter_values must contain a JSON object."
    )


print("Parameter parsing completed")
print("------------------------------------")
print(f"Parameters supplied : {len(parsed_parameters)}")


if parsed_parameters:
    for parameter_name, parameter_value in parsed_parameters.items():
        print(
            f"{parameter_name} : "
            f"{parameter_value}"
        )
else:
    print("No parameters supplied.")


# --------------------------------------------------
# 3. Read parameter definition from rules_m
# --------------------------------------------------

parameter_definition_obj = None


if parameter_definition not in (None, "", "null", "NULL"):

    try:
        parameter_definition_obj = json.loads(
            parameter_definition
        )

    except json.JSONDecodeError as e:
        raise ValueError(
            f"parameter_definition for rule_code='{rule_code}' "
            "is not valid JSON.\n"
            f"Error: {str(e)}"
        )


# --------------------------------------------------
# 4. Validate required parameters
# --------------------------------------------------

if parameter_definition_obj:

    if isinstance(parameter_definition_obj, dict):

        parameter_definitions = (
            parameter_definition_obj.get(
                "parameters",
                []
            )
        )

    elif isinstance(parameter_definition_obj, list):

        parameter_definitions = parameter_definition_obj

    else:

        parameter_definitions = []


    for parameter in parameter_definitions:

        if not isinstance(parameter, dict):
            continue

        parameter_name = parameter.get("name")
        required = parameter.get("required", False)

        if (
            required
            and parameter_name
            and parameter_name not in parsed_parameters
        ):
            raise ValueError(
                f"Required parameter '{parameter_name}' "
                f"is missing for rule_code='{rule_code}'."
            )


print("")
print("Parameter validation PASSED")

PARSING RULE PARAMETERS
Parameter parsing completed
------------------------------------
Parameters supplied : 3
source_timestamp_column : INITIATED_AT
ingestion_timestamp_column : SETTLED_AT
time_unit : MINUTE

Parameter validation PASSED


In [0]:
# ============================================================
# VALIDATE REQUIRED PARAMETERS
# ============================================================

print("====================================")
print("VALIDATING RULE PARAMETERS")
print("====================================")

required_parameters = []

if parameter_definition:
    try:
        parameter_definition_json = json.loads(
            parameter_definition
        )

        required_parameters = [
            p["name"]
            for p in parameter_definition_json.get("parameters", [])
            if p.get("required") is True
        ]

    except Exception as e:
        raise ValueError(
            f"Invalid parameter_definition JSON for rule "
            f"{rule_code}: {e}"
        )


missing_parameters = []

for parameter_name in required_parameters:

    value = parsed_parameters.get(parameter_name)

    if value is None:
        missing_parameters.append(parameter_name)

    elif isinstance(value, str) and not value.strip():
        missing_parameters.append(parameter_name)


if missing_parameters:

    raise ValueError(
        f"Rule {rule_code} is missing required parameters: "
        f"{', '.join(missing_parameters)}"
    )


print("Required parameters : ", required_parameters)
print("Provided parameters : ", list(parsed_parameters.keys()))
print("Parameter validation PASSED")

VALIDATING RULE PARAMETERS
Required parameters :  ['source_timestamp_column', 'ingestion_timestamp_column', 'time_unit']
Provided parameters :  ['source_timestamp_column', 'ingestion_timestamp_column', 'time_unit']
Parameter validation PASSED


In [0]:
# ============================================================
# BUILD EXECUTION CONTEXT
# ============================================================

print("====================================")
print("BUILDING EXECUTION CONTEXT")
print("====================================")


# ------------------------------------------------------------
# 1. Source relation
# ------------------------------------------------------------

source_relation = SOURCE_VIEW


# ------------------------------------------------------------
# 2. Safely quote target column
# ------------------------------------------------------------

target_column_sql = None

if target_column_name is not None:

    target_column_sql = (
        "`"
        + target_column_name.replace("`", "``")
        + "`"
    )


# ------------------------------------------------------------
# 3. Resolve effective threshold
#
# Precedence:
#   1. Notebook input
#   2. Assignment threshold
#   3. Rule default threshold
# ------------------------------------------------------------

effective_threshold = None

if (
    threshold_value is not None
    and str(threshold_value).strip() != ""
):

    effective_threshold = float(
        str(threshold_value).strip()
    )

elif assignment["threshold_value"] is not None:

    effective_threshold = float(
        assignment["threshold_value"]
    )

elif default_threshold is not None:

    effective_threshold = float(
        default_threshold
    )


# ------------------------------------------------------------
# 4. Validate effective threshold
# ------------------------------------------------------------

if effective_threshold is None:

    raise ValueError(
        f"No effective threshold found for rule_code='{rule_code}'. "
        f"assignment_id={assignment_id_int}, "
        f"version_number={version_number_int}. "
        f"Checked notebook input threshold_value, "
        f"rules_assignments_m.threshold_value, "
        f"and rules_m.default_threshold."
    )


# ------------------------------------------------------------
# 5. Resolve effective evaluation operation
#
# Precedence:
#   1. Notebook input
#   2. Assignment value
# ------------------------------------------------------------

effective_evaluation_operation = None

if (
    evaluation_operation is not None
    and str(evaluation_operation).strip() != ""
):

    effective_evaluation_operation = (
        str(evaluation_operation).strip()
    )

elif assignment["evaluation_operation"] is not None:

    effective_evaluation_operation = (
        str(assignment["evaluation_operation"]).strip()
    )


# ------------------------------------------------------------
# 6. Validate effective evaluation operation
# ------------------------------------------------------------

if not effective_evaluation_operation:

    raise ValueError(
        f"No effective evaluation operation found for "
        f"rule_code='{rule_code}'. "
        f"assignment_id={assignment_id_int}."
    )


# ------------------------------------------------------------
# 7. Resolve effective SQL expression
# ------------------------------------------------------------

effective_sql_expression = None

if (
    sql_expression is not None
    and str(sql_expression).strip() != ""
):

    effective_sql_expression = (
        str(sql_expression).strip()
    )

elif assignment["sql_expression"] is not None:

    effective_sql_expression = (
        str(assignment["sql_expression"]).strip()
    )


# ------------------------------------------------------------
# 8. Build execution context
# ------------------------------------------------------------

execution_context = {

    "table": source_relation,

    "asset": source_relation,

    "column": target_column_sql,

    "asset_id": asset_id_int,

    "asset_name": asset_name,

    "asset_type": asset_type,

    "assignment_id": assignment_id_int,

    "rule_code": rule_code,

    "version_number": version_number_int,

    "dimension": dimension,

    "rule_name": rule_name,

    "rule_type": rule_type,

    "target_type": target_type,

    "target_column_id": target_column_id_int,

    "target_column_name": target_column_name,

    "target_column_sql": target_column_sql,

    "source_relation": source_relation,

    "parameter_values": parsed_parameters,

    "sql_expression": effective_sql_expression,

    "threshold_value": effective_threshold,

    "evaluation_operation": effective_evaluation_operation,

    "data_type": data_type,

    "semantic_type": semantic_type
}


# ------------------------------------------------------------
# 9. Derived placeholders
# ------------------------------------------------------------

if "required_columns" in parsed_parameters:

    required_columns = parsed_parameters["required_columns"]

    if not isinstance(required_columns, list):
        raise ValueError(
            "required_columns must be a JSON array of column names."
        )

    if len(required_columns) == 0:
        raise ValueError(
            "required_columns cannot be empty."
        )

    required_conditions = []

    for col in required_columns:

        required_conditions.append(
            f"`{col}` IS NOT NULL "
            f"AND TRIM(CAST(`{col}` AS STRING)) <> ''"
        )

    execution_context["required_columns_condition"] = (
        " AND ".join(required_conditions)
    )


if "selected_columns" in parsed_parameters:

    selected_columns = parsed_parameters["selected_columns"]

    if not isinstance(selected_columns, list):
        raise ValueError(
            "selected_columns must be a JSON array of column names."
        )

    if len(selected_columns) == 0:
        raise ValueError(
            "selected_columns cannot be empty."
        )

    selected_conditions = []

    for col in selected_columns:

        selected_conditions.append(
            f"`{col}` IS NOT NULL "
            f"AND TRIM(CAST(`{col}` AS STRING)) <> ''"
        )

    execution_context["selected_columns_condition"] = (
        " AND ".join(selected_conditions)
    )


if "date_sequence_columns" in parsed_parameters:

    date_sequence_columns = (
        parsed_parameters["date_sequence_columns"]
    )

    if not isinstance(date_sequence_columns, list):
        raise ValueError(
            "date_sequence_columns must be a JSON array of column names."
        )

    if len(date_sequence_columns) < 2:
        raise ValueError(
            "date_sequence_columns must contain at least two columns."
        )

    date_conditions = []

    for i in range(len(date_sequence_columns) - 1):

        current_column = date_sequence_columns[i]

        next_column = date_sequence_columns[i + 1]

        date_conditions.append(
            f"`{current_column}` <= `{next_column}`"
        )

    execution_context["date_sequence_condition"] = (
        " AND ".join(date_conditions)
    )


# ------------------------------------------------------------
# 10. Identify date/time parameter columns
# ------------------------------------------------------------
#
# Any parameterized rule whose metadata data_type is
# DATE/TIMESTAMP can contain column_select controls that
# represent timestamp columns.
#
# We retain the original parameter names so the renderer
# can safely convert those values to TRY_CAST expressions.
#
# ------------------------------------------------------------

date_time_parameter_names = []

if (
    data_type is not None
    and str(data_type).strip().upper()
    in (
        "DATE",
        "TIMESTAMP",
        "DATE/TIMESTAMP"
    )
):

    if parameter_definition_obj:

        if isinstance(
            parameter_definition_obj,
            dict
        ):

            parameter_definitions = (
                parameter_definition_obj.get(
                    "parameters",
                    []
                )
            )

        elif isinstance(
            parameter_definition_obj,
            list
        ):

            parameter_definitions = (
                parameter_definition_obj
            )

        else:

            parameter_definitions = []

        for parameter in parameter_definitions:

            if not isinstance(parameter, dict):
                continue

            parameter_name = parameter.get("name")

            control = parameter.get(
                "control",
                "text"
            )

            if (
                parameter_name
                and control in (
                    "column_select",
                    "column"
                )
                and parameter_name in parsed_parameters
            ):

                date_time_parameter_names.append(
                    parameter_name
                )


execution_context[
    "date_time_parameter_names"
] = date_time_parameter_names


# ------------------------------------------------------------
# 11. Display resolved values
# ------------------------------------------------------------

print("")
print("EXECUTION VALUES")
print("------------------------------------")

print(
    f"Source relation          : {source_relation}"
)

print(
    f"Target column            : {target_column_name}"
)

print(
    f"Effective threshold      : {effective_threshold}"
)

print(
    f"Evaluation operation     : "
    f"{effective_evaluation_operation}"
)

print(
    f"Date/time parameters     : "
    f"{date_time_parameter_names}"
)


print("")
print("Execution context created")
print("------------------------------------")

for key, value in execution_context.items():

    if key == "parameter_values":

        print(
            f"{key:25} : "
            f"{json.dumps(value, default=str)}"
        )

    else:

        print(
            f"{key:25} : {value}"
        )

BUILDING EXECUTION CONTEXT

EXECUTION VALUES
------------------------------------
Source relation          : dq_source_data
Target column            : None
Effective threshold      : 60.0
Evaluation operation     : <=
Date/time parameters     : ['source_timestamp_column', 'ingestion_timestamp_column']

Execution context created
------------------------------------
table                     : dq_source_data
asset                     : dq_source_data
column                    : None
asset_id                  : 28
asset_name                : 01_TXN_transaction_lifecycle
asset_type                : TABLE
assignment_id             : 35
rule_code                 : T03
version_number            : 1
dimension                 : TIMELINESS
rule_name                 : Average latency
rule_type                 : PARAMETERIZED
target_type               : TABLE / COLUMN
target_column_id          : None
target_column_name        : None
target_column_sql         : None
source_relation           : dq_s

In [0]:
# ============================================================
# RENDER SQL TEMPLATE
# ============================================================

print("====================================")
print("RENDERING SQL TEMPLATE")
print("====================================")


# ------------------------------------------------------------
# 1. Validate SQL template
# ------------------------------------------------------------

if not sql_query_template:

    raise ValueError(
        f"No sql_query_template found for rule {rule_code}."
    )


# ------------------------------------------------------------
# 2. SQL helper functions
# ------------------------------------------------------------

def sql_string(value):

    if value is None:
        return "NULL"

    value = str(value)

    return (
        "'"
        + value.replace("'", "''")
        + "'"
    )


def sql_identifier(value):

    if value is None:

        raise ValueError(
            "SQL identifier cannot be NULL."
        )

    value = str(value).strip()

    if not value:

        raise ValueError(
            "SQL identifier cannot be empty."
        )

    return (
        "`"
        + value.replace("`", "``")
        + "`"
    )


def sql_number(value):

    if value is None:

        raise ValueError(
            "Numeric parameter cannot be NULL."
        )

    value_string = str(value).strip()

    if not re.fullmatch(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)",
        value_string
    ):

        raise ValueError(
            f"Invalid numeric SQL value: {value_string}"
        )

    return value_string


def sql_multi_value(value):

    if value is None:

        raise ValueError(
            "Multi-value parameter cannot be NULL."
        )

    if not isinstance(value, list):

        raise ValueError(
            "Multi-value parameter must be a JSON array."
        )

    if len(value) == 0:

        raise ValueError(
            "Multi-value parameter cannot be empty."
        )

    return ", ".join(
        sql_string(v)
        for v in value
    )


def sql_timestamp_identifier(value):

    """
    Render a source timestamp column as a safe timestamp
    expression.

    TRY_CAST is intentional.

    Invalid source values become NULL instead of terminating
    the entire DQ rule execution.
    """

    return (
        f"TRY_CAST("
        f"{sql_identifier(value)} "
        f"AS TIMESTAMP)"
    )


# ------------------------------------------------------------
# 3. Start with metadata SQL template
# ------------------------------------------------------------

rendered_sql = sql_query_template


# ------------------------------------------------------------
# 4. Standard placeholders
# ------------------------------------------------------------

standard_placeholders = {

    "{table}": source_relation,

    "{asset}": source_relation,

    "{column}": target_column_sql
}


for token, value in standard_placeholders.items():

    if token in rendered_sql:

        if value is None:

            raise ValueError(
                f"Placeholder {token} requires a value."
            )

        rendered_sql = rendered_sql.replace(
            token,
            str(value)
        )


# ------------------------------------------------------------
# 5. Parameterized placeholders
# ------------------------------------------------------------

if parameter_definition_obj:

    if isinstance(
        parameter_definition_obj,
        dict
    ):

        parameter_definitions = (
            parameter_definition_obj.get(
                "parameters",
                []
            )
        )

    elif isinstance(
        parameter_definition_obj,
        list
    ):

        parameter_definitions = (
            parameter_definition_obj
        )

    else:

        parameter_definitions = []


    for parameter in parameter_definitions:

        if not isinstance(parameter, dict):
            continue

        parameter_name = parameter.get("name")

        control = parameter.get(
            "control",
            "text"
        )

        if not parameter_name:
            continue

        value = parsed_parameters.get(
            parameter_name
        )

        token = (
            "{"
            + parameter_name
            + "}"
        )


        # ----------------------------------------------------
        # Placeholder may be represented by a derived
        # execution-context value instead.
        # ----------------------------------------------------

        if token not in rendered_sql:
            continue


        # ----------------------------------------------------
        # Column parameter
        # ----------------------------------------------------

        if control in (
            "column_select",
            "column"
        ):

            if (
                parameter_name
                in date_time_parameter_names
            ):

                rendered_value = (
                    sql_timestamp_identifier(
                        value
                    )
                )

            else:

                rendered_value = sql_identifier(
                    value
                )


        # ----------------------------------------------------
        # Multiple columns
        # ----------------------------------------------------

        elif control in (
            "column_multi_select",
            "ordered_column_select"
        ):

            if not isinstance(
                value,
                list
            ):

                raise ValueError(
                    f"Parameter '{parameter_name}' "
                    f"must be a JSON array."
                )

            if len(value) == 0:

                raise ValueError(
                    f"Parameter '{parameter_name}' "
                    f"cannot be empty."
                )


            if (
                data_type is not None
                and str(data_type).strip().upper()
                in (
                    "DATE",
                    "TIMESTAMP",
                    "DATE/TIMESTAMP"
                )
            ):

                rendered_value = ", ".join(
                    sql_timestamp_identifier(v)
                    for v in value
                )

            else:

                rendered_value = ", ".join(
                    sql_identifier(v)
                    for v in value
                )


        # ----------------------------------------------------
        # Numeric parameter
        # ----------------------------------------------------

        elif control == "number":

            rendered_value = sql_number(
                value
            )


        # ----------------------------------------------------
        # String / regex / datetime
        # ----------------------------------------------------

        elif control in (
            "text",
            "regex",
            "datetime"
        ):

            rendered_value = sql_string(
                value
            )


        # ----------------------------------------------------
        # Multi-value parameter
        # ----------------------------------------------------

        elif control == "multi_value":

            rendered_value = sql_multi_value(
                value
            )


        # ----------------------------------------------------
        # Raw SQL expression
        # ----------------------------------------------------

        elif control in (
            "expression",
            "expression_multi"
        ):

            if value is None:

                raise ValueError(
                    f"Parameter '{parameter_name}' "
                    f"cannot be NULL."
                )

            rendered_value = str(value)


        # ----------------------------------------------------
        # Select parameter
        # ----------------------------------------------------

        elif control == "select":

            if value is None:

                raise ValueError(
                    f"Parameter '{parameter_name}' "
                    f"cannot be NULL."
                )

            rendered_value = str(value)


        # ----------------------------------------------------
        # Asset selection
        # ----------------------------------------------------

        elif control == "asset_select":

            if value is None:

                raise ValueError(
                    f"Parameter '{parameter_name}' "
                    f"cannot be NULL."
                )

            rendered_value = str(value)


        # ----------------------------------------------------
        # JSON / schema definition
        # ----------------------------------------------------

        elif control in (
            "json",
            "schema_definition"
        ):

            rendered_value = sql_string(
                json.dumps(
                    value,
                    default=str
                )
            )


        # ----------------------------------------------------
        # Default
        # ----------------------------------------------------

        else:

            rendered_value = sql_string(
                value
            )


        rendered_sql = rendered_sql.replace(
            token,
            rendered_value
        )


# ------------------------------------------------------------
# 6. Replace derived placeholders
# ------------------------------------------------------------

for placeholder, value in execution_context.items():

    token = (
        "{"
        + placeholder
        + "}"
    )

    if token not in rendered_sql:
        continue

    if value is None:

        raise ValueError(
            f"Placeholder {token} requires a value."
        )

    # Do not replace parameters a second time.
    if placeholder in parsed_parameters:
        continue

    # date_time_parameter_names is execution metadata,
    # not a SQL placeholder.
    if placeholder == "date_time_parameter_names":
        continue

    rendered_sql = rendered_sql.replace(
        token,
        str(value)
    )


# ------------------------------------------------------------
# 7. Check for unresolved placeholders
# ------------------------------------------------------------

unresolved_placeholders = re.findall(
    r"\{([A-Za-z0-9_]+)\}",
    rendered_sql
)


if unresolved_placeholders:

    raise ValueError(
        f"Unresolved SQL placeholders for rule "
        f"{rule_code}: "
        f"{unresolved_placeholders}"
    )


# ------------------------------------------------------------
# 8. Display final SQL
# ------------------------------------------------------------

print("")
print("RENDERED SQL")
print("------------------------------------")

print(rendered_sql)

print("")
print("SQL rendering completed")
print("------------------------------------")

print(rendered_sql)

RENDERING SQL TEMPLATE

RENDERED SQL
------------------------------------

SELECT
    CAST(
        AVG(
            TIMESTAMPDIFF(
                MINUTE,
                TRY_CAST(`INITIATED_AT` AS TIMESTAMP),
                TRY_CAST(`SETTLED_AT` AS TIMESTAMP)
            )
        )
        AS DOUBLE
    ) AS metric_value,

    0 AS passed_count,
    0 AS failed_count,
    COUNT(*) AS total_count,
    COUNT(*) AS applicable_count
FROM dq_source_data
WHERE TRY_CAST(`INITIATED_AT` AS TIMESTAMP) IS NOT NULL
  AND TRY_CAST(`SETTLED_AT` AS TIMESTAMP) IS NOT NULL


SQL rendering completed
------------------------------------

SELECT
    CAST(
        AVG(
            TIMESTAMPDIFF(
                MINUTE,
                TRY_CAST(`INITIATED_AT` AS TIMESTAMP),
                TRY_CAST(`SETTLED_AT` AS TIMESTAMP)
            )
        )
        AS DOUBLE
    ) AS metric_value,

    0 AS passed_count,
    0 AS failed_count,
    COUNT(*) AS total_count,
    COUNT(*) AS applicable_count
FROM dq

In [0]:
# ============================================================
# EXECUTE RULE SQL
# ============================================================

print("====================================")
print("EXECUTING RULE SQL")
print("====================================")


# --------------------------------------------------
# 1. Execute rendered SQL
# --------------------------------------------------

try:

    rule_result_df = spark.sql(
        rendered_sql
    )

except Exception as e:

    raise RuntimeError(
        f"Rule SQL execution failed for "
        f"rule_code='{rule_code}'.\n"
        f"Error: {str(e)}"
    )


# --------------------------------------------------
# 2. Validate result structure
# --------------------------------------------------

result_columns = rule_result_df.columns

required_result_columns = [
    "metric_value",
    "passed_count",
    "failed_count",
    "total_count",
    "applicable_count"
]


missing_result_columns = [
    column
    for column in required_result_columns
    if column not in result_columns
]


if missing_result_columns:

    raise ValueError(
        "Rule SQL did not return the required "
        f"standard result columns: "
        f"{missing_result_columns}\n"
        f"Returned columns: {result_columns}"
    )


# --------------------------------------------------
# 3. Validate result row count
# --------------------------------------------------

result_row_count = rule_result_df.count()


if result_row_count != 1:

    raise ValueError(
        "Rule SQL must return exactly one result row. "
        f"Returned {result_row_count} rows."
    )


rule_result = rule_result_df.first()


# --------------------------------------------------
# 4. Extract standardized metrics
# --------------------------------------------------

metric_value = (
    float(rule_result["metric_value"])
    if rule_result["metric_value"] is not None
    else None
)


passed_count = (
    int(rule_result["passed_count"])
    if rule_result["passed_count"] is not None
    else 0
)


failed_count = (
    int(rule_result["failed_count"])
    if rule_result["failed_count"] is not None
    else 0
)


total_count = (
    int(rule_result["total_count"])
    if rule_result["total_count"] is not None
    else 0
)


applicable_count = (
    int(rule_result["applicable_count"])
    if rule_result["applicable_count"] is not None
    else 0
)


# --------------------------------------------------
# 5. Identify malformed DATE/TIMESTAMP values
# --------------------------------------------------
#
# This is performed against the RAW source view.
#
# NULL values are NOT malformed.
#
# A value is malformed when:
#
#   raw value IS NOT NULL
#   AND TRY_CAST(raw value AS TIMESTAMP) IS NULL
#
# For a rule containing multiple timestamp columns,
# a row is malformed when ANY required timestamp column
# is malformed.
#
# --------------------------------------------------

malformed_count = 0
date_time_total_count = 0
date_time_applicable_count = 0


if date_time_parameter_names:

    malformed_conditions = []

    applicable_conditions = []

    for parameter_name in date_time_parameter_names:

        source_column = parsed_parameters.get(
            parameter_name
        )

        if source_column is None:
            continue

        quoted_column = sql_identifier(
            source_column
        )

        malformed_conditions.append(
            "("
            f"{quoted_column} IS NOT NULL "
            f"AND TRY_CAST("
            f"{quoted_column} AS TIMESTAMP"
            f") IS NULL"
            ")"
        )

        applicable_conditions.append(
            f"{quoted_column} IS NOT NULL"
        )


    # --------------------------------------------------
    # A row is malformed if ANY required timestamp
    # column is malformed.
    # --------------------------------------------------

    if malformed_conditions:

        malformed_condition_sql = (
            " OR ".join(
                malformed_conditions
            )
        )

        malformed_row_df = spark.sql(
            f"""
            SELECT COUNT(*) AS malformed_count
            FROM {SOURCE_VIEW}
            WHERE {malformed_condition_sql}
            """
        )

        malformed_count = int(
            malformed_row_df.first()["malformed_count"]
        )


    # --------------------------------------------------
    # Count rows where all required timestamp columns
    # are non-null.
    # --------------------------------------------------

    if applicable_conditions:

        applicable_condition_sql = (
            " AND ".join(
                applicable_conditions
            )
        )

        date_time_applicable_df = spark.sql(
            f"""
            SELECT COUNT(*) AS applicable_count
            FROM {SOURCE_VIEW}
            WHERE {applicable_condition_sql}
            """
        )

        date_time_applicable_count = int(
            date_time_applicable_df.first()[
                "applicable_count"
            ]
        )


    # --------------------------------------------------
    # Count rows where all required timestamp columns
    # successfully convert.
    # --------------------------------------------------

    valid_conditions = []

    for parameter_name in date_time_parameter_names:

        source_column = parsed_parameters.get(
            parameter_name
        )

        if source_column is None:
            continue

        quoted_column = sql_identifier(
            source_column
        )

        valid_conditions.append(
            f"TRY_CAST("
            f"{quoted_column} AS TIMESTAMP"
            f") IS NOT NULL"
        )


    if valid_conditions:

        valid_condition_sql = (
            " AND ".join(
                valid_conditions
            )
        )

        valid_timestamp_df = spark.sql(
            f"""
            SELECT COUNT(*) AS valid_count
            FROM {SOURCE_VIEW}
            WHERE {valid_condition_sql}
            """
        )

        date_time_total_count = int(
            valid_timestamp_df.first()[
                "valid_count"
            ]
        )


# --------------------------------------------------
# 6. Add malformed values to DQ failures
# --------------------------------------------------
#
# IMPORTANT:
#
# Malformed timestamp records are DQ failures.
#
# They are not counted as applicable records because
# the timestamp-based rule cannot calculate a metric
# for those rows.
#
# --------------------------------------------------

if date_time_parameter_names:

    failed_count = (
        failed_count
        + malformed_count
    )


# --------------------------------------------------
# 7. Calculate failure percentage
# --------------------------------------------------

if total_count > 0:

    failure_percentage = (
        100.0
        * failed_count
        / total_count
    )

else:

    failure_percentage = 0.0


# --------------------------------------------------
# 8. Display result
# --------------------------------------------------

print("Rule SQL executed successfully")
print("------------------------------------")

print(
    f"metric_value             : "
    f"{metric_value}"
)

print(
    f"passed_count             : "
    f"{passed_count}"
)

print(
    f"failed_count             : "
    f"{failed_count}"
)

print(
    f"total_count              : "
    f"{total_count}"
)

print(
    f"applicable_count         : "
    f"{applicable_count}"
)

print(
    f"failure_percentage       : "
    f"{failure_percentage:.2f}"
)

print("")
print("DATE/TIMESTAMP QUALITY")
print("------------------------------------")

print(
    f"date/time parameters     : "
    f"{date_time_parameter_names}"
)

print(
    f"valid timestamp rows     : "
    f"{date_time_total_count}"
)

print(
    f"non-null timestamp rows  : "
    f"{date_time_applicable_count}"
)

print(
    f"malformed timestamp rows : "
    f"{malformed_count}"
)

EXECUTING RULE SQL
Rule SQL executed successfully
------------------------------------
metric_value             : -383942.47252747254
passed_count             : 0
failed_count             : 2
total_count              : 91
applicable_count         : 91
failure_percentage       : 2.20

DATE/TIMESTAMP QUALITY
------------------------------------
date/time parameters     : ['source_timestamp_column', 'ingestion_timestamp_column']
valid timestamp rows     : 91
non-null timestamp rows  : 93
malformed timestamp rows : 2


In [0]:
# ============================================================
# EVALUATE RULE STATUS
# ============================================================

print("====================================")
print("EVALUATING RULE RESULT")
print("====================================")


# --------------------------------------------------
# 1. Validate evaluation operation
# --------------------------------------------------

valid_operations = {
    ">",
    ">=",
    "<",
    "<=",
    "=",
    "==",
    "!="
}


if (
    effective_evaluation_operation
    not in valid_operations
):

    raise ValueError(
        f"Unsupported evaluation operation: "
        f"{effective_evaluation_operation}"
    )


# --------------------------------------------------
# 2. Evaluate metric against threshold
# --------------------------------------------------

if metric_value is None:

    metric_evaluation_passed = False

else:

    if effective_evaluation_operation == ">":

        metric_evaluation_passed = (
            metric_value
            > effective_threshold
        )

    elif effective_evaluation_operation == ">=":

        metric_evaluation_passed = (
            metric_value
            >= effective_threshold
        )

    elif effective_evaluation_operation == "<":

        metric_evaluation_passed = (
            metric_value
            < effective_threshold
        )

    elif effective_evaluation_operation == "<=":

        metric_evaluation_passed = (
            metric_value
            <= effective_threshold
        )

    elif effective_evaluation_operation in (
        "=",
        "=="
    ):

        metric_evaluation_passed = (
            metric_value
            == effective_threshold
        )

    elif effective_evaluation_operation == "!=":

        metric_evaluation_passed = (
            metric_value
            != effective_threshold
        )


# --------------------------------------------------
# 3. Apply malformed-data failure condition
# --------------------------------------------------
#
# A rule cannot be considered fully PASS when the
# underlying data contains malformed values required
# by the rule.
#
# --------------------------------------------------

malformed_data_failed = (
    malformed_count > 0
)


# --------------------------------------------------
# 4. Determine final rule status
# --------------------------------------------------

if malformed_data_failed:

    evaluation_status = "FAIL"

elif metric_evaluation_passed:

    evaluation_status = "PASS"

else:

    evaluation_status = "FAIL"


# --------------------------------------------------
# 5. Display evaluation
# --------------------------------------------------

print("Rule evaluation completed")
print("------------------------------------")

print(
    f"Metric value              : "
    f"{metric_value}"
)

print(
    f"Threshold                 : "
    f"{effective_threshold}"
)

print(
    f"Evaluation operation      : "
    f"{effective_evaluation_operation}"
)

print(
    f"Metric evaluation         : "
    f"{'PASS' if metric_evaluation_passed else 'FAIL'}"
)

print(
    f"Malformed data rows       : "
    f"{malformed_count}"
)

print(
    f"Malformed data condition : "
    f"{'FAIL' if malformed_data_failed else 'PASS'}"
)

print(
    f"Final rule status         : "
    f"{evaluation_status}"
)

EVALUATING RULE RESULT
Rule evaluation completed
------------------------------------
Metric value              : -383942.47252747254
Threshold                 : 60.0
Evaluation operation      : <=
Metric evaluation         : PASS
Malformed data rows       : 2
Malformed data condition : FAIL
Final rule status         : FAIL


In [0]:
print("====================================")
print("WRITING RULE EXECUTION RESULT")
print("====================================")


# --------------------------------------------------
# 1. Build result details
# --------------------------------------------------

result_details = {
    "rule_code": rule_code,
    "rule_name": rule_name,
    "version_number": version_number_int,
    "target_column_id": target_column_id_int,
    "target_column_name": target_column_name,

    "metric_value": metric_value,
    "threshold_value": effective_threshold,
    "evaluation_operation": effective_evaluation_operation,

    "passed_count": passed_count,
    "failed_count": failed_count,
    "total_count": total_count,
    "applicable_count": applicable_count,

    "parameter_values": parsed_parameters
}


result_details_json = json.dumps(
    result_details,
    default=str
)


# --------------------------------------------------
# 2. Check for asset execution context
# --------------------------------------------------

asset_execution_id = globals().get(
    "asset_execution_id",
    None
)


if asset_execution_id in (
    None,
    "",
    "None"
):

    print(
        "No asset_execution_id supplied."
    )

    print(
        "Standalone test mode: "
        "result will not be inserted into "
        "dq_execution_results_m yet."
    )

    result_written = False


else:

    try:

        asset_execution_id_int = int(
            asset_execution_id
        )

    except ValueError:

        raise ValueError(
            "asset_execution_id must be a valid integer."
        )


    # --------------------------------------------------
    # 3. Prepare values
    # --------------------------------------------------

    failure_percentage_value = (
        Decimal(str(round(failure_percentage, 2)))
        if failure_percentage is not None
        else None
    )

    threshold_value_value = (
        Decimal(str(effective_threshold))
        if effective_threshold is not None
        else None
    )


    # --------------------------------------------------
    # 4. Insert using explicit target columns
    #
    # result_id is an IDENTITY column and is therefore
    # intentionally excluded.
    #
    # created_at is generated by the table default.
    # --------------------------------------------------

    result_df = spark.createDataFrame(
        [
            (
                asset_execution_id_int,
                rule_code,
                None,                         # rule_version_id - legacy field
                assignment_id_int,
                dimension,
                evaluation_status,
                int(total_count) if total_count is not None else None,
                int(passed_count) if passed_count is not None else None,
                int(failed_count) if failed_count is not None else None,
                failure_percentage_value,
                threshold_value_value,
                result_details_json
            )
        ],
        T.StructType([
            T.StructField(
                "asset_execution_id",
                T.LongType(),
                False
            ),
            T.StructField(
                "rule_code",
                T.StringType(),
                True
            ),
            T.StructField(
                "rule_version_id",
                T.LongType(),
                True
            ),
            T.StructField(
                "assignment_id",
                T.LongType(),
                True
            ),
            T.StructField(
                "dimension",
                T.StringType(),
                False
            ),
            T.StructField(
                "status",
                T.StringType(),
                False
            ),
            T.StructField(
                "records_evaluated",
                T.LongType(),
                True
            ),
            T.StructField(
                "records_passed",
                T.LongType(),
                True
            ),
            T.StructField(
                "records_failed",
                T.LongType(),
                True
            ),
            T.StructField(
                "failure_percentage",
                T.DecimalType(7, 2),
                True
            ),
            T.StructField(
                "threshold_value",
                T.DecimalType(18, 6),
                True
            ),
            T.StructField(
                "result_details",
                T.StringType(),
                True
            )
        ])
    )


    # --------------------------------------------------
    # 5. Write only the 12 result columns
    # --------------------------------------------------

    result_df.createOrReplaceTempView(
        "dq_rule_result_to_write"
    )


    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.dq_execution_results_m
        (
            asset_execution_id,
            rule_code,
            rule_version_id,
            assignment_id,
            dimension,
            status,
            records_evaluated,
            records_passed,
            records_failed,
            failure_percentage,
            threshold_value,
            result_details
        )
        SELECT
            asset_execution_id,
            rule_code,
            rule_version_id,
            assignment_id,
            dimension,
            status,
            records_evaluated,
            records_passed,
            records_failed,
            failure_percentage,
            threshold_value,
            result_details
        FROM dq_rule_result_to_write
    """)


    result_written = True

    print(
        "Result successfully written to "
        "dq_execution_results_m."
    )


# --------------------------------------------------
# 6. Display result summary
# --------------------------------------------------

print("")
print("RESULT SUMMARY")
print("------------------------------------")

print(f"Rule code          : {rule_code}")
print(f"Assignment ID      : {assignment_id_int}")
print(f"Status             : {evaluation_status}")
print(f"Metric value       : {metric_value}")
print(f"Threshold          : {effective_threshold}")
print(f"Records evaluated  : {total_count}")
print(f"Records passed     : {passed_count}")
print(f"Records failed     : {failed_count}")
print(f"Result written     : {result_written}")

WRITING RULE EXECUTION RESULT
No asset_execution_id supplied.
Standalone test mode: result will not be inserted into dq_execution_results_m yet.

RESULT SUMMARY
------------------------------------
Rule code          : T03
Assignment ID      : 35
Status             : FAIL
Metric value       : -383942.47252747254
Threshold          : 60.0
Records evaluated  : 91
Records passed     : 0
Records failed     : 2
Result written     : False


In [0]:
# ============================================================
# RETURN RESULT TO ORCHESTRATOR
# ============================================================

result = {
    "status": evaluation_status,
    "rule_code": rule_code,
    "assignment_id": assignment_id_int,
    "metric_value": metric_value,
    "threshold_value": effective_threshold,
    "evaluation_operation": effective_evaluation_operation,
    "records_evaluated": total_count,
    "records_passed": passed_count,
    "records_failed": failed_count,
    "failure_percentage": failure_percentage,
    "result_written": result_written
}

print("")
print("====================================")
print("RULE EXECUTION COMPLETE")
print("====================================")

print(json.dumps(result, default=str, indent=2))

dbutils.notebook.exit(
    json.dumps(result, default=str)
)